# 04 SIPRI

**Owner:** Yair

**What this notebook does:** turns the SIPRI trade register into arms imports per country-year and the top 3 suppliers per country.

**Reads:** `trade-register.csv`, `crosswalk_sipri.csv`

**Writes:** `sipri_cy.csv`, `sipri_suppliers.csv`

This is a main notebook. It must run from top to bottom with Kernel → Restart and Run All.
Write simple code in small steps: one operation per line, print the result after each step. Your sandbox section for this notebook: **sandbox_yair, Part A**. Paste only the working lines, one step per cell, with a short note above each.

## Setup

In [18]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RAW = "../data/raw/"
CLEAN = "../data/clean/"
pd.set_option("display.max_columns", 50)

## A1 Load the trade register

Paste your working code from the sandbox. Add one sentence above it that says what it does.

we see the rough shape of the dataset, how many columns we are going to have and a couple exmaple rows

In [19]:
# your code here
sipri_raw = pd.read_csv(RAW + "trade-register.csv", skiprows=11, encoding="latin-1")
print(sipri_raw.shape)
print(sipri_raw.columns)
print(sipri_raw.head(3))

(62009, 17)
Index(['SIPRI AT Database ID', 'Supplier', 'Recipient', 'Designation',
       'Description', 'Armament category', 'Order date',
       'Order date is estimate', 'Deliveries in the Year Range',
       'Deliveries in the Year Range is estimate', 'Delivery year',
       'Delivery year is estimate', 'Status', 'SIPRI estimate',
       'TIV deal unit', 'TIV delivery values', 'Local production'],
      dtype='str')
   SIPRI AT Database ID      Supplier    Recipient  Designation  \
0                 53791        Russia  Afghanistan        Mi-17   
1                 33640        Russia  Afghanistan        Mi-17   
2                 16639  Soviet Union  Afghanistan  BM-13 132mm   

                Description Armament category  Order date  \
0      transport helicopter          Aircraft        2010   
1      transport helicopter          Aircraft        2004   
2  multiple rocket launcher         Artillery        1960   

  Order date is estimate  Deliveries in the Year Range  \
0   

## A2 Rename and keep what you need

Paste your working code from the sandbox. Add one sentence above it that says what it does.

renamed to column to fit our other datasets/create shared column names, and delete the columns we wont need

In [20]:
# your code here
new_names = {"Supplier": "supplier", "Recipient": "recipient", "Delivery year": "year", "TIV delivery values": "tiv", "Armament category": "category", "Status": "status"}
sipri_raw_renamed = sipri_raw.rename(columns=new_names)

keep = []
for x in new_names:
    keep.append(new_names[x])
sipri = sipri_raw_renamed[keep].copy()
print(sipri)

sipri["recipient"] = sipri["recipient"].str.strip()




           supplier    recipient    year    tiv           category  \
0            Russia  Afghanistan  2011.0  17.40           Aircraft   
1            Russia  Afghanistan  2005.0  11.60           Aircraft   
2      Soviet Union  Afghanistan  1960.0   2.00          Artillery   
3      Soviet Union  Afghanistan  1961.0   2.00          Artillery   
4      Soviet Union  Afghanistan  1979.0  12.50  Armoured vehicles   
...             ...          ...     ...    ...                ...   
62004         China     Zimbabwe  1986.0   3.00           Aircraft   
62005         China     Zimbabwe  1991.0  18.00           Aircraft   
62006         Italy     Zimbabwe  1967.0   9.00          Artillery   
62007        France     Zimbabwe  1977.0   1.05           Aircraft   
62008        Russia     Zimbabwe  1999.0  21.60           Aircraft   

            status  
0      Second hand  
1      Second hand  
2      Second hand  
3      Second hand  
4      Second hand  
...            ...  
62004  Secon

## A3 Drop undelivered deals and non-states

Paste your working code from the sandbox. Add one sentence above it that says what it does.

we removed deals that were not delivered

we removed all the non-state suppliers and the unknown suppliers, aswell as adjusting the year format

In [21]:
# your code here
before = len(sipri)
sipri = sipri.dropna(subset=["year"]).copy()
print("removed no-year rows:", before - len(sipri))

sipri["year"] = sipri["year"].astype(int)

before = len(sipri)
print("non-state recipients:")
print(sipri[sipri["recipient"].str.endswith("*", na=False)]["recipient"].unique())
sipri = sipri[~sipri["recipient"].str.endswith("*", na=False)].copy()
print("removed star rows:", before - len(sipri))

before = len(sipri)
unknown = sipri[sipri["recipient"].str.contains("unknown")]
print("unknown recipients:")
print(unknown["recipient"].unique())
sipri = sipri[~sipri["recipient"].str.contains("unknown", case=False, na=False)].copy()
print("removed unknown rows:", before - len(sipri))
print("rows left:", len(sipri))



removed no-year rows: 1220
non-state recipients:
<ArrowStringArray>
[                              'African Union**',
                               'Amal (Lebanon)*',
                           'ANC (South Africa)*',
                    'Anti-Castro rebels (Cuba)*',
                            'Armas (Guatemala)*',
                          'Contras (Nicaragua)*',
                        'Darfur rebels (Sudan)*',
                               'ELF (Ethiopia)*',
                              'EPLF (Ethiopia)*',
                                   'FAN (Chad)*',
                           'FMLN (El Salvador)*',
                                'FNLA (Angola)*',
                           'FRELIMO (Portugal)*',
                                  'GUNT (Chad)*',
                                 'Haiti rebels*',
                            'Hamas (Palestine)*',
                          'Hezbollah (Lebanon)*',
             'House of Representatives (Libya)*',
                        'Houthi 

## A4 Add iso3

Paste your working code from the sandbox. Add one sentence above it that says what it does.

In [22]:
crosswalk_sipri = pd.read_csv(CLEAN + "crosswalk_sipri.csv", encoding="latin-1")
print(crosswalk_sipri.shape)
print(crosswalk_sipri.columns)
print(crosswalk_sipri.head(3))
rows_before_merge = len(sipri)

sipri = pd.merge(sipri, crosswalk_sipri, on="recipient", how="left",validate="many_to_one", indicator=True )

print(len(sipri) - rows_before_merge, "missing rows")

print(sipri["_merge"].value_counts())
print((sipri["_merge"] == "left_only").sum())
lost = sipri[sipri["_merge"] == "left_only"].copy()
novdem = sorted(lost["recipient"].unique())
print(novdem)

for country in novdem:
    print(country)
    print("Rows:", len(sipri[sipri["recipient"] == country]))
    print("Total tiv:", sipri[sipri["recipient"] == country]["tiv"].sum())

sipri = sipri[sipri["_merge"] == "both"].copy()
sipri = sipri.drop(columns="_merge")



(188, 2)
Index(['recipient', 'iso3'], dtype='str')
     recipient iso3
0  Afghanistan  AFG
1      Albania  ALB
2      Algeria  DZA
0 missing rows
_merge
both          59608
left_only       571
right_only        0
Name: count, dtype: int64
571
['Biafra', 'Czechoslovakia', 'Katanga', 'Palestine', 'Tonga', 'Western Sahara']
Biafra
Rows: 6
Total tiv: 8.9
Czechoslovakia
Rows: 522
Total tiv: 29400.489999999998
Katanga
Rows: 6
Total tiv: 9.52
Palestine
Rows: 12
Total tiv: 39.98
Tonga
Rows: 7
Total tiv: 32.290000000000006
Western Sahara
Rows: 18
Total tiv: 78.8


## A5 Imports per country-year

Paste your working code from the sandbox. Add one sentence above it that says what it does.

we cleaned the sipri data: 
summed up all same year imports per country into one row.
removed years with no imports



In [23]:

india_2000_hand = sipri[(sipri["recipient"] == "India") & (sipri["year"] == 2000)
                        ]["tiv"].sum()
print(india_2000_hand)

# your code here
sipri_cy = (sipri.groupby(["iso3", "year"], as_index=False).agg(arms_imports_tiv=("tiv", "sum"))).copy()

india_2000 = sipri_cy[(sipri_cy["iso3"] == "IND") & (sipri_cy["year"] == 2000)]

print(india_2000)


987.53
     iso3  year  arms_imports_tiv
3286  IND  2000            987.53


## A6 Top suppliers and save

Paste your working code from the sandbox. Add one sentence above it that says what it does.

In [24]:
india = sipri[sipri["iso3"] == "IND"].copy()
india_totals = (
    india.groupby("supplier", as_index=False)
    .agg(tiv=("tiv", "sum"))
    .sort_values("tiv", ascending=False)
)
india_top3 = india_totals.head(3)["supplier"].tolist()
print(india_top3)

top3 = {}
for iso, rows in sipri.groupby("iso3"):
    totals = (
        rows.groupby("supplier", as_index=False)
        .agg(tiv=("tiv", "sum"))
        .sort_values("tiv", ascending=False)
    )
    top3[iso] = list(totals.head(3)["supplier"])
print(top3["IND"])

supplier_group = []
for iso, supplier in zip(sipri["iso3"], sipri["supplier"]):
    if supplier in top3[iso]:
        supplier_group.append(supplier)
    else:
        supplier_group.append("Other")

sipri["supplier_group"] = supplier_group

sipri_suppliers = (
    sipri.groupby(["iso3", "year", "supplier_group"], as_index=False)
    .agg(tiv=("tiv", "sum"))
    .rename(columns={"supplier_group": "supplier"})
    .copy()
)

sipri_cy.to_csv(CLEAN + "sipri_cy.csv", index=False)
sipri_suppliers.to_csv(CLEAN + "sipri_suppliers.csv", index=False)


['Russia', 'Soviet Union', 'United Kingdom']
['Russia', 'Soviet Union', 'United Kingdom']


**Checkpoint.** Run this cell. OK means done. Do not change it.

In [25]:
sipri_cy_check = pd.read_csv(CLEAN + "sipri_cy.csv")
sup_check = pd.read_csv(CLEAN + "sipri_suppliers.csv")
assert list(sipri_cy_check.columns) == ["iso3", "year", "arms_imports_tiv"]
assert list(sup_check.columns) == ["iso3", "year", "supplier", "tiv"]
assert sipri_cy_check.duplicated(["iso3", "year"]).sum() == 0
assert sup_check.groupby("iso3")["supplier"].nunique().max() <= 4
assert abs(sup_check["tiv"].sum() - sipri_cy_check["arms_imports_tiv"].sum()) < 0.01
print("OK")

OK


## Decisions in this notebook

Write every choice you made here (and copy it into the decisions log in `CONVENTIONS.md`). You need these for the report.

| Decision | Why |
| --- | --- |
|  |  |